In [30]:
import os
import csv
import sys
from datetime import datetime, timedelta
import requests
from bs4 import BeautifulSoup

# 콘솔 출력 인코딩 설정 (Windows 환경의 한글 깨짐 방지)
if sys.platform.startswith('win'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except AttributeError:
        pass

In [31]:
# 주요 부동산 정책 데이터 정의 (정부, 핵심 정책, 발표일, 분석용 대표 시행일, 정책 성격)
POLICIES = [
    {
        "president": "박근혜",
        "policy": "4·1 부동산 대책",
        "announcement_date": "2013-04-01",
        "effective_date": "2013-04-22",
        "summary": "거래 활성화·규제 완화"
    },
    {
        "president": "박근혜",
        "policy": "11·3 부동산 대책",
        "announcement_date": "2016-11-03",
        "effective_date": "2016-11-15",
        "summary": "과열 억제·청약 규제 강화"
    },
    {
        "president": "문재인",
        "policy": "8·2 부동산 대책",
        "announcement_date": "2017-08-02",
        "effective_date": "2017-08-23",
        "summary": "투기 억제·LTV/DTI 강화"
    },
    {
        "president": "문재인",
        "policy": "12·16 부동산 대책",
        "announcement_date": "2019-12-16",
        "effective_date": "2019-12-17",
        "summary": "고가주택·대출 규제 강화"
    },
    {
        "president": "윤석열",
        "policy": "8·16 국민 주거안정 실현방안",
        "announcement_date": "2022-08-16",
        "effective_date": "2022-08-16",  # 단일 시행일 없음 -> 발표일 사용
        "summary": "270만 호 공급·정비사업 개선"
    },
    {
        "president": "윤석열",
        "policy": "1·3 부동산 규제완화",
        "announcement_date": "2023-01-03",
        "effective_date": "2023-01-05",
        "summary": "서울 대부분 규제지역 해제"
    },
    {
        "president": "이재명",
        "policy": "6·27 가계부채 관리 강화방안",
        "announcement_date": "2025-06-27",
        "effective_date": "2025-06-28",
        "summary": "수도권 주담대 규제 강화"
    },
    {
        "president": "이재명",
        "policy": "10·15 주택시장 안정화 대책",
        "announcement_date": "2025-10-15",
        "effective_date": "2025-10-20",
        "summary": "토허제·규제지역·대출규제 강화"
    }
]

In [32]:
def get_policy_mapping(date_str):
    """
    입력된 날짜에 대응하는 부동산 정책 정보 및 시기(시행전, 시행일, 초기반응, 체감반응)를 조회합니다.

    Args:
        date_str (str): YYYYMMDD 또는 YYYY-MM-DD 형식의 날짜 문자열

    Returns:
        list: 매칭된 정책 정보 딕셔너리 리스트. 매칭되는 정책이 없을 경우 빈 리스트를 반환합니다.
    """
    # 날짜 파싱 시도
    input_date = None
    for fmt in ("%Y%m%d", "%Y-%m-%d"):
        try:
            input_date = datetime.strptime(date_str, fmt).date()
            break
        except ValueError:
            continue

    if not input_date:
        print(f"[Warning] 날짜 포맷이 올바르지 않습니다: {date_str}")
        return []

    matched = []

    for item in POLICIES:
        eff_date = datetime.strptime(item["effective_date"], "%Y-%m-%d").date()

        # 시기 정의 계산
        before_start = eff_date - timedelta(days=30)
        before_end = eff_date - timedelta(days=1)
        after_start = eff_date + timedelta(days=1)
        after_end = eff_date + timedelta(days=30)
        feel_start = eff_date + timedelta(days=31)
        feel_end = eff_date + timedelta(days=90)

        period = None
        if before_start <= input_date <= before_end:
            period = "시행전"
        elif input_date == eff_date:
            period = "시행일"
        elif after_start <= input_date <= after_end:
            period = "초기반응"
        elif feel_start <= input_date <= feel_end:
            period = "체감반응"

        if period:
            matched.append({
                "president": item["president"],
                "policy": item["policy"],
                "policy_summary": item["summary"],
                "period": period,
                "date": input_date.strftime("%Y-%m-%d")
            })

    return matched

In [33]:
def scrape_naver_land_news(date_str):
    """
    네이버 부동산 뉴스 섹션에서 특정 날짜의 기사 제목과 URL을 최대 20개 스크래핑합니다.

    Args:
        date_str (str): YYYYMMDD 형식의 날짜 문자열 (예: '20260819')

    Returns:
        list: (기사제목, URL) 튜플 리스트
    """
    url = f"https://news.naver.com/breakingnews/section/101/260?date={date_str}"
    
    headers = {
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'
    }

    try:
        res = requests.get(url, headers=headers)
        if not res.ok:
            print(f"[Error] 네이버 뉴스 스크래핑 실패. 상태 코드: {res.status_code}")
            return []
            
        soup = BeautifulSoup(res.text, 'html.parser')
        a_tags = soup.select("a.sa_text_title")
        
        articles = []
        for a_tag in a_tags[:20]:  # 최대 20개 제한
            title = a_tag.text.strip()
            link = a_tag.get('href', '').strip()
            if title and link:
                articles.append((title, link))
                
        return articles
        
    except Exception as e:
        print(f"[Error] 스크래핑 진행 중 예외 발생: {e}")
        return []

In [34]:
def save_to_csv(rows, filename="News_Scraping.csv"):
    """
    가공된 뉴스 데이터를 CSV 파일에 누적 저장하며, 중복된 데이터는 저장하지 않습니다.
    
    중복 판단 기준은 (정책명, 날짜, 기사 URL)의 조합입니다.

    Args:
        rows (list): [대통령, 정책, 정책요약, 시기, 날짜, 기사제목, url, 감정] 형태의 데이터 리스트
        filename (str): 저장할 CSV 파일 이름
    """
    file_exists = os.path.exists(filename)
    existing_keys = set()
    
    # 1. 파일이 이미 존재하면 기존에 등록된 키 값을 읽어서 중복 판단용 set을 구축
    if file_exists:
        try:
            with open(filename, mode='r', encoding='utf-8-sig') as f:
                reader = csv.reader(f)
                header = next(reader, None)
                if header:
                    for line in reader:
                        if len(line) >= 7:
                            policy = line[1]
                            date = line[4]
                            url = line[6]
                            existing_keys.add((policy, date, url))
        except Exception as e:
            print(f"[Warning] 기존 CSV 파일을 읽는 동안 오류가 발생했습니다: {e}")
            
    # 2. 이번에 추가할 행들 중 중복되지 않는 고유한 행만 선별
    new_rows = []
    for r in rows:
        policy = r[1]
        date = r[4]
        url = r[6]
        key = (policy, date, url)
        if key not in existing_keys:
            new_rows.append(r)
            existing_keys.add(key)
            
    if not new_rows:
        print(f"[Info] 추가할 새로운 뉴스 데이터가 없습니다 (모든 데이터가 '{filename}'에 이미 존재함).")
        return

    # 3. 새로운 행들만 추가 저장
    try:
        file_exists = os.path.exists(filename)
        with open(filename, mode='a', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(["대통령", "정책", "정책요약", "시기", "날짜", "기사제목", "url", "감정"])
            
            writer.writerows(new_rows)
        print(f"[Success] {len(new_rows)}개의 새로운 행이 '{filename}'에 성공적으로 추가 저장되었습니다.")
    except Exception as e:
        print(f"[Error] CSV 파일 저장 중 예외 발생: {e}")


def sort_csv_by_date(filename="News_Scraping.csv"):
    """
    저장된 CSV 파일을 날짜 순으로 정렬하여 덮어씁니다.
    날짜 필드(index 4)를 기준으로 오름차순 정렬을 수행합니다.

    Args:
        filename (str): 정렬할 CSV 파일 이름
    """
    if not os.path.exists(filename):
        print(f"[Warning] 정렬할 파일이 존재하지 않습니다: {filename}")
        return

    try:
        # 1. 기존 데이터 읽기
        with open(filename, mode='r', encoding='utf-8-sig') as f:
            reader = csv.reader(f)
            header = next(reader, None)
            if not header:
                print(f"[Info] 정렬 대상 CSV 파일이 비어 있습니다: {filename}")
                return
            rows = list(reader)

        # 2. 날짜 필드(index 4) 기준 정렬
        def parse_date(row):
            if len(row) > 4:
                try:
                    return datetime.strptime(row[4], "%Y-%m-%d").date()
                except ValueError:
                    pass
            return datetime.min.date()

        rows.sort(key=parse_date)

        # 3. 정렬된 데이터로 파일 덮어쓰기
        with open(filename, mode='w', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(header)
            writer.writerows(rows)
            
        print(f"[Success] '{filename}' 파일이 날짜 순으로 정렬되었습니다. (총 {len(rows)}개 행)")
    except Exception as e:
        print(f"[Error] CSV 파일 정렬 중 예외 발생: {e}")

In [35]:
def run_scraping_flow(date_str, filename="News_Scraping.csv"):
    """
    특정 날짜에 대한 전체 스크래핑 및 매핑, CSV 저장 과정을 처리하는 메인 흐름 제어 함수입니다.

    Args:
        date_str (str): YYYYMMDD 또는 YYYY-MM-DD 형식의 날짜 문자열
        filename (str): 저장할 CSV 파일 이름
    """
    # YYYY-MM-DD 또는 YYYYMMDD 포맷 정규화
    clean_date_str = date_str.replace("-", "")
    
    # 1. 정책 정보 매핑 판별
    policy_mappings = get_policy_mapping(clean_date_str)
    
    # 2. 뉴스 스크래핑
    articles = scrape_naver_land_news(clean_date_str)
    if not articles:
        print(f"[Info] {clean_date_str} 날짜에 스크래핑된 뉴스가 없습니다.")
        return
        
    # 3. 데이터 가공
    rows = []
    if policy_mappings:
        for mapping in policy_mappings:
            for title, url in articles:
                rows.append([
                    mapping["president"],
                    mapping["policy"],
                    mapping["policy_summary"],
                    mapping["period"],
                    mapping["date"],
                    title,
                    url,
                    ""
                ])
    else:
        formatted_date = f"{clean_date_str[:4]}-{clean_date_str[4:6]}-{clean_date_str[6:8]}"
        for title, url in articles:
            rows.append([
                "", "", "", "", formatted_date, title, url, ""
            ])
            
    # 4. CSV 저장 및 정렬
    save_to_csv(rows, filename)
    sort_csv_by_date(filename)

In [36]:
def scrape_all_policies(sample_mode=True, filename="News_Scraping.csv"):
    """
    정의된 모든 부동산 정책에 대해 시기별 뉴스를 수집합니다.
    
    Args: 
        sample_mode (bool): True이면 각 정책의 시기(시행전, 시행일, 초기반응, 체감반응)별로 
                            대표 날짜를 1일씩만 샘플링하여 빠르게 수집합니다.
                            False이면 정책별 전체 기간(총 121일/정책)의 모든 날짜를 수집합니다. (시간이 오래 걸립니다)
        filename (str): 저장할 CSV 파일 이름
    """
    import time
    total_started = time.time()
    print(f"\n[Start] 모든 부동산 정책 데이터 수집을 시작합니다. (샘플 모드: {sample_mode})")
    
    tasks = []
    
    for p in POLICIES:
        eff_date = datetime.strptime(p["effective_date"], "%Y-%m-%d").date()
        
        # 각 시기별 날짜 리스트 정의
        periods = {
            "시행전": [eff_date - timedelta(days=i) for i in range(30, 0, -1)],
            "시행일": [eff_date],
            "초기반응": [eff_date + timedelta(days=i) for i in range(1, 31)],
            "체감반응": [eff_date + timedelta(days=i) for i in range(31, 91)]
        }
        
        for period_name, date_list in periods.items():
            if sample_mode:
                sampled_date = date_list[len(date_list) // 2]
                tasks.append((sampled_date, p))
            else:
                for d in date_list:
                    tasks.append((d, p))
                    
    total_tasks = len(tasks)
    print(f"[Info] 총 수집 대상 일수: {total_tasks}일")
    
    success_count = 0
    for idx, (target_date, policy_info) in enumerate(tasks, 1):
        date_str = target_date.strftime("%Y%m%d")
        print(f"[{idx}/{total_tasks}] {policy_info['policy']} ({target_date.strftime('%Y-%m-%d')}) 수집 중...", end=" ", flush=True)
        
        articles = scrape_naver_land_news(date_str)
        if articles:
            rows = []
            mappings = get_policy_mapping(date_str)
            
            if not mappings:
                mappings = [{
                    "president": policy_info["president"],
                    "policy": policy_info["policy"],
                    "policy_summary": policy_info["summary"],
                    "period": "알수없음",
                    "date": target_date.strftime("%Y-%m-%d")
                }]
                
            for mapping in mappings:
                for title, url in articles:
                    rows.append([
                        mapping["president"],
                        mapping["policy"],
                        mapping["policy_summary"],
                        mapping.get("period", "알수없음"),
                        mapping["date"],
                        title,
                        url,
                        ""
                    ])
            if rows:
                save_to_csv(rows, filename)
                success_count += 1
        else:
            print("[Warning] 스크래핑 기사 없음")
        
        delay = 0.3 if sample_mode else 1.0
        time.sleep(delay)
        
    duration = time.time() - total_started
    print(f"\n[Finished] 수집 완료. 소요 시간: {duration:.2f}초. 성공 일수: {success_count}/{total_tasks}")
    
    # 수집 완료 후 자동으로 날짜 순 정렬 수행
    sort_csv_by_date(filename)

In [ ]:
# 1. 전체 정책에 대해 시기별로 1일씩 샘플링하여 수집 및 CSV 저장을 실행합니다.
scrape_all_policies(sample_mode=True)

# 2. (참고) 만약 스크래핑 없이 기존 파일의 정렬만 따로 실행하고 싶다면 아래 함수를 사용
# sort_csv_by_date()


[Start] 모든 부동산 정책 데이터 수집을 시작합니다. (샘플 모드: True)
[Info] 총 수집 대상 일수: 32일
[1/32] 4·1 부동산 대책 (2013-04-07) 수집 중... [Success] 20개의 새로운 행이 'News_Scraping.csv'에 성공적으로 추가 저장되었습니다.
[2/32] 4·1 부동산 대책 (2013-04-22) 수집 중... [Success] 20개의 새로운 행이 'News_Scraping.csv'에 성공적으로 추가 저장되었습니다.
[3/32] 4·1 부동산 대책 (2013-05-08) 수집 중... [Success] 20개의 새로운 행이 'News_Scraping.csv'에 성공적으로 추가 저장되었습니다.
[4/32] 4·1 부동산 대책 (2013-06-22) 수집 중... [Success] 20개의 새로운 행이 'News_Scraping.csv'에 성공적으로 추가 저장되었습니다.
[5/32] 11·3 부동산 대책 (2016-10-31) 수집 중... [Success] 20개의 새로운 행이 'News_Scraping.csv'에 성공적으로 추가 저장되었습니다.
[6/32] 11·3 부동산 대책 (2016-11-15) 수집 중... [Success] 20개의 새로운 행이 'News_Scraping.csv'에 성공적으로 추가 저장되었습니다.
[7/32] 11·3 부동산 대책 (2016-12-01) 수집 중... [Success] 20개의 새로운 행이 'News_Scraping.csv'에 성공적으로 추가 저장되었습니다.
[8/32] 11·3 부동산 대책 (2017-01-15) 수집 중... [Success] 20개의 새로운 행이 'News_Scraping.csv'에 성공적으로 추가 저장되었습니다.
[9/32] 8·2 부동산 대책 (2017-08-08) 수집 중... [Success] 20개의 새로운 행이 'News_Scraping.csv'에 성공적으로 추가 저장되었습니다.
[10/32] 8·2 부동산 대책 (2017-

: 